# 🔥 guato_ba — Modelos de Deep Learning
### 7 arquiteturas de segmentação bitemporal + a função de perda
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Os modelos DL fazem **segmentação semântica bitemporal**: recebem o par de datas (T1 e T2) e devolvem, para **cada pixel do patch**, um *logit* de "queimou / não queimou". Todas as arquiteturas vivem em `redes.py` e são construídas por `construir_dl(nome)`.

Há duas famílias de entrada:
- **Concat** (`in_ch=8`): empilha as 4 bandas de T1 e T2 → a rede vê as 8 camadas juntas.
- **Siamesa** (`in_ch=6`): um mesmo codificador processa T1 e T2 (3 bandas cada, sem o vermelho) e a rede compara as duas representações.

</div>

# <span style="color:#cf4c39">🧰 1. As 7 configurações</span>
<hr style="border:1px solid #e69627;">

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT/"redes.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch, redes
print("PyTorch:", torch.__version__)

bandas_nome = {0:"B04", 1:"B8A", 2:"B11", 3:"B12"}
print(f"\n{'modelo':15s} {'classe':18s} {'in_ch':>5s}  {'micro-batch':>11s}  bandas")
for nome, cfg in redes.MODELOS_DL.items():
    bns = ",".join(bandas_nome[b] for b in cfg["bandas"])
    print(f"{nome:15s} {cfg['cls'].__name__:18s} {cfg['in_ch']:>5d}  {cfg['micro']:>11d}  {bns}")

PyTorch: 2.8.0+cu128

modelo          classe             in_ch  micro-batch  bandas
unet            UNet                   8          256  B04,B8A,B11,B12
unet_nored      UNet                   6          256  B8A,B11,B12
att_unet        AttUNet                8          128  B04,B8A,B11,B12
biunet          BiUNet                 6          128  B8A,B11,B12
biatt_unet      BiAttUNet              6          128  B8A,B11,B12
resnet34_siam   ResNet34SiamUNet       6           64  B8A,B11,B12
deeplabv3plus   DeepLabV3Plus          8           96  B04,B8A,B11,B12


<div style="text-align: justify; margin-right: 2.5%;">

`in_ch` = bandas × 2 datas. `unet_nored` é a U-Net **sem a banda vermelha** (isola o efeito do vermelho/nuvem). O **micro-batch** é quanto cabe na GPU de uma vez; o batch efetivo (280) vem de **acumulação de gradiente**.

</div>

# <span style="color:#cf4c39">🔢 2. Quantos parâmetros tem cada rede</span>
<hr style="border:1px solid #e69627;">

Tamanho de cada arquitetura (medido com `pretrained=False`):

| modelo | parâmetros | entrada |
|---|---|---|
| `unet` | 17,46 M | concat (8) |
| `unet_nored` | 17,46 M | concat (6) |
| `att_unet` | 18,06 M | concat (8) |
| `deeplabv3plus` | 19,08 M | concat (8) |
| `biunet` | 20,40 M | siamesa (6) |
| `biatt_unet` | 21,97 M | siamesa (6) |
| `resnet34_siam` | 26,30 M | siamesa (6), backbone ImageNet |


<div style="text-align: justify; margin-right: 2.5%;">

A célula reconstrói cada modelo e confirma os números no seu ambiente. O `resnet34_siam` precisa de **torchvision** (backbone ResNet34); se ele não estiver instalado, o modelo é apenas pulado — os demais rodam normalmente.

</div>

In [2]:
x8 = torch.randn(2, 8, 128, 128)   # entrada concat
x6 = torch.randn(2, 6, 128, 128)   # entrada siamesa
print(f"{'modelo':15s} {'saída':>18s}  {'parâmetros':>11s}")
for nome, cfg in redes.MODELOS_DL.items():
    try:
        m = redes.construir_dl(nome, p=0.2, pretrained=False).eval()
        x = x8 if cfg["in_ch"] == 8 else x6
        with torch.no_grad():
            y = m(x)
        n = sum(p.numel() for p in m.parameters())/1e6
        print(f"{nome:15s} {str(tuple(y.shape)):>18s}  {n:>9.2f} M")
    except ModuleNotFoundError as e:
        print(f"{nome:15s} {'(pulado: '+str(e)+')':>18s}")

modelo                       saída   parâmetros


unet                 (2, 128, 128)      17.46 M


unet_nored           (2, 128, 128)      17.46 M


att_unet             (2, 128, 128)      18.06 M


biunet               (2, 128, 128)      20.40 M


biatt_unet           (2, 128, 128)      21.97 M


resnet34_siam        (2, 128, 128)      26.30 M


deeplabv3plus        (2, 128, 128)      19.08 M


# <span style="color:#cf4c39">🎯 3. A função de perda: Focal Tversky + BCE de borda</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Queimada é **rara**: uma perda ingênua ignoraria a classe positiva. Por isso usamos `make_loss()`, que soma dois termos:

1. **Focal Tversky** (dominante) — uma Dice/IoU generalizada que **penaliza mais os falsos negativos** (`tv_b=0.7` > `tv_a=0.3`) e foca nos casos difíceis (`tv_g=0.75`). É o que empurra o IoU da queimada para cima.
2. **BCE ponderada por borda** — entropia cruzada com **peso maior nas bordas** da cicatriz (`border_w=4`), para contornos mais nítidos.

Os pixels `255` (**inválidos**) entram com `ign=255` e são **removidos** do cálculo. Vamos ver a perda funcionando num exemplo sintético.

</div>

In [3]:
import torch.nn.functional as F
loss_fn = redes.make_loss(border_w=4.0, tv_a=0.3, tv_b=0.7, tv_g=0.75, w_bce=0.5, ign=255)

torch.manual_seed(0)
logit  = torch.randn(2, 64, 64)                      # saída crua do modelo (B,H,W)
target = torch.zeros(2, 64, 64)
target[:, 20:40, 20:40] = 1                          # uma "cicatriz" quadrada
target[:, :5, :]        = 255                         # faixa inválida (ignorada)
pw = torch.tensor(5.0)                               # pos_weight (classe rara)

perda_ruim = loss_fn(logit, target, pw)              # modelo aleatório
perda_boa  = loss_fn((target.clamp(0,1)*8 - 4), target, pw)   # "modelo" quase perfeito
print(f"perda (predição aleatória): {perda_ruim.item():.3f}")
print(f"perda (predição quase certa): {perda_boa.item():.3f}   <- bem menor, como esperado")

perda (predição aleatória): 1.618
perda (predição quase certa): 0.133   <- bem menor, como esperado


# <span style="color:#cf4c39">⚙️ 4. A receita de treino (resumo)</span>
<hr style="border:1px solid #e69627;">

- **Otimizador:** AdamW (`lr=3e-4`, `weight_decay=1e-4`), `CosineAnnealingWarmRestarts`, AMP (precisão mista), dropout 0,2.
- **Padronização:** z-score por canal, estimado **só no treino** (média/desvio salvos no checkpoint).
- **Augmentation:** espelhamentos (H/V) + rotações de 90°.
- **Amostragem:** batch estratificado de **280 patches** (~12,8% queimada), cada patch visto 1×/época.
- **Split:** 70% treino / 30% validação, estratificado; *early-stop* pelo **IoU de validação**.
- **Normalização de camada:** `GroupNorm` (variável de ambiente `GUATO_NORM=gn`) elimina o descompasso treino/inferência de batches pequenos.

Para treinar de verdade: `python main.py train-dl --modelo unet`. A busca de hiperparâmetros está no notebook **07**.

---
Continue no **`03_modelos_machine_learning`** para a abordagem por pixel.